### **Extraer el `nombre del video`, `usuario`, `tiempo del comentario`, `likes del comentario` y `comentarios` de múltiples videos de una playlist de `Youtube`**

Vamos a scrapear el siguiente sitio:

https://www.youtube.httcom/playlist?list=PLuaGRMrO-j-8NndtkHMA7Y_7798tdJWKH

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/DwWM7f8k/ws-334.png"></center>

Obtener todos los videos listados en la playlist

<center><img src="https://i.postimg.cc/5NrsKcpP/ws-335.png"></center>

Obtengo el enlace de cada video. Luego con un for trabajaremos con cada URL

<center><img src="https://i.postimg.cc/Dy0CcfQJ/ws-336.png"></center>

Ya desde la pagina, localizo cuantos comentarios hay en total segun youtube:

<center><img src="https://i.postimg.cc/908Ljs2Q/ws-338.png"></center>

Localizamos el titulo del video:

<center><img src="https://i.postimg.cc/3xDnPLc3/ws-344.png"></center>

Localizar la caja que contiene todos los articulos listados en la página (Este tag aparece cuando NO SCROLLEO hacia bien abajo en la página)

<center><img src="https://i.postimg.cc/fbmq0Hc8/ws-339.png"></center>

(Este tag aparece cuando SI SCROLLEO hacia bien abajo en la página, pues eso es lo que hace el script de Javascript que escribimos, por tanto, este es el que debo usar):

<center><img src="https://i.postimg.cc/ZnkHH8TV/ws-346.png"></center>

Localizar la caja que contiene los comentarios (aparece cuando NO SCROLLEO)

<center><img src="https://i.postimg.cc/QCxfwhRz/ws-342.png"></center>

(Este tag aparece cuando SI SCROLLEO, por tanto, este debo usar)

<center><img src="https://i.postimg.cc/vHWyry7d/ws-347.png"></center>

Localizar el nombre del usuario que escribió el comentario

<center><img src="https://i.postimg.cc/Y9xXdhyg/ws-340.png"></center>

Localizar el tiempo que tiene el comentario (aparece cuando NO SCROLLEO)

<center><img src="https://i.postimg.cc/T2C7m5bv/ws-341.png"></center>

(Este tag aparece cuando SI SCROLLEO, por tanto, este debo usar)

<center><img src="https://i.postimg.cc/yxKH5rBG/ws-348.png"></center>

Localizar el numero de 'Me gusta' que tiene el comentario

<center><img src="https://i.postimg.cc/LXSCkzv2/ws-343.png"></center>

#### **Código**

In [ ]:
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
#from webdriver_manager.chrome import ChromeDriverManager # pip install webdriver-manager

# window.scrollTo(): Con este método, puede controlar las coordenadas de la posición de desplazamiento de la página. El método 
# window.scrollTo() toma dos argumentos: el primer argumento es una coordenada x y el segundo es una coordenada y. Cuando se llama 
# a este método, se desplaza el documento a las coordenadas especificadas.

# Funcion para obtener el Script de Scrolling dependiendo de cuantos scrollings ya he hecho
# Mientras mas scrolls llevo dando, mas pixeles voy bajando.
# Simplemente remplazo el 20000 en la cadena del script, por un numero que dependa de la iteracion en que me encuentro actualmente
def obtener_script_scrolling(iteration): 
    scrollingScript = """ 
      window.scrollTo(0, 20000)
    """
    return scrollingScript.replace('20000', str(20000 * (iteration + 1)))


options = Options()
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

#URL SEMILLA
website = "https://www.youtube.com/playlist?list=PLuaGRMrO-j-8NndtkHMA7Y_7798tdJWKH"
# Se recomienda utilizar Chrome, pero podriamos utilizar Firefox, Safari, Edge, etc.
# driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()), options=options)
driver = webdriver.Chrome(options=options)
driver.get(website)
driver.maximize_window()

# Dejamos que el sitio cargue por 2 segundos
sleep(6)

# Pulsar el boton 'No, gracias' para suscripcion de Youtube Premium
try:
    btn = driver.find_element(by=By.XPATH, value='//button[@class="yt-spec-button-shape-next yt-spec-button-shape-next--text yt-spec-button-shape-next--mono yt-spec-button-shape-next--size-m"]')
    btn.click()
except:
    pass

# Inicializando el almacenamiento de los enlaces de los videos
videos_url = []

# Obtener todos los videos listados en la playlist
videos = driver.find_elements(By.XPATH, '//div[@id="contents"]/ytd-playlist-video-renderer')

for video in videos:
    videos_url.append(video.find_element(By.XPATH, './/h3/a[@id="video-title"]').get_attribute("href"))

# Inicializando el almacenamiento de la data
nombre_video = []
usuario_comentario = []
tiempo_comentario = []
texto_comentario = []
likes_comentario = [] 

# Vamos a cada video
for url in videos_url:

    # Comienzo a trabajar con cada uno de los enlaces de los videos de la playlist
    driver.get(url)
    sleep(3)

    # Hago un primer scrolling hacia abajo para que cargue el numero de comentarios
    driver.execute_script("""window.scrollTo(0, 400)""")
    sleep(3)

    # Obtengo cuantos comentarios hay en total segun youtube
    # Aproximo cuantos deberian de haber (ya que youtube oculta algunos)
    num_comentarios_totales = driver.find_element(by=By.XPATH, value='//h2[@id="count"]//span[1]').text
    num_comentarios_totales = int(num_comentarios_totales) * 0.90

    # Captar el nombre del video
    titulo_video = driver.find_element(by=By.XPATH, value='//h1[@class="style-scope ytd-watch-metadata"]').text  

    # Vamos a captar el numero de comentarios cargados hasta el momento
    # En nuestro caso capta 20 comentarios inmediatamente
    comentarios_cargados = len(driver.find_elements(by=By.XPATH, value='//ytd-comment-thread-renderer'))

    # Cada vez que hago scrolling voy a comparar si es que ya se encuentran cargados todos los comentarios que youtube dice que hay 
    n_scrolls = 0
    n_scrolls_maximo = 10

    while comentarios_cargados < num_comentarios_totales and n_scrolls < n_scrolls_maximo:
        driver.execute_script(obtener_script_scrolling(n_scrolls))
        n_scrolls += 1
        sleep(3)
        comentarios_cargados = len(driver.find_elements(By.XPATH, '//ytd-comment-thread-renderer'))   

    # Localizar la caja que contiene todos los articulos listados en la página
    comentarios = driver.find_elements(by=By.XPATH, value='//ytd-comment-thread-renderer') 

    # Recorrer la lista de comentarios
    for comentario in comentarios:

        nombre_video.append(titulo_video)
        # Localizar el nombre del usuario que escribió el comentario
        usuario_comentario.append(comentario.find_element(by=By.XPATH, value='.//a[@id="author-text"]').text)
        # Localizar el tiempo que tiene el comentario
        tiempo_comentario.append(comentario.find_element(by=By.XPATH, value='.//yt-formatted-string[@class="published-time-text style-scope ytd-comment-renderer"]').text)  
        # Localizar el numero de 'Me gusta' que tiene el comentario
        likes_comentario.append(comentario.find_element(by=By.XPATH, value='.//span[@id="vote-count-middle"]').text)         
        # Localizar el comentario
        texto_comentario.append(comentario.find_element(by=By.XPATH, value='.//yt-formatted-string[@id="content-text"]').text) 

# Almacenar los datos en un DataFrame y exportar a un archivo CSV
df = pd.DataFrame({'titulo': nombre_video, 'usuario': usuario_comentario, 'tiempo':tiempo_comentario, 'me_gusta':likes_comentario, 'comentario':texto_comentario})
df.to_csv(f'comentarios_playlist.csv', index=False)  

driver.quit()